# Boiling Point (bp) — ANN (Artificial Neural Network) Analysis

This notebook builds an ANN (Multi-Layer Perceptron) model using AV-AM matrix
descriptors ONLY for the bp (boiling point) property, and compares it with
the MLR results in `MLR_Analysis.ipynb`.

**Scope deliberately narrowed**: in the earlier study, ANN was applied not to
all 12 properties but only to bp (this notebook returns to that scope). bp is
the MOST COMPLETE property in the data set (n=66, no missing data at all) -
the most favorable property for a method like ANN that requires more
parameters.

**The earlier ANN study was not consulted at all** - this notebook was
designed from scratch to directly address the criticisms in the reviewer
reports (data leakage ambiguity, lack of hyperparameter justification,
missing OECD validation criteria, non-reproducible reporting of early
stopping).

**How it works**: Run the cells below in order. ANN will be trained and
validated for all 6 groups, and a single Word report (`bp_ANN_Analysis.docx`)
will contain both the ANN results and the comparison with MLR.

**IMPORTANT**: Set the `EXCEL_PATH` variable to your own file path.
Running this notebook can take a VERY LONG TIME (on the order of hours) -
this is normal and a DELIBERATE choice: unlike MLR, which is computed
instantly (VIF/OLS), ANN needs to be trained separately for each
hyperparameter combination; moreover, this entire nested cross-validation
process (hyperparameter search, final model, and the 1000 trials of
Y-randomization included) is repeated FROM SCRATCH with SEED_COUNT (10)
different seeds to guard against the randomness of the neural network's
weight initialization, and the results are reported as mean±std (see the
Method Description, section 3). You only need to keep your
computer/Colab session open/active for the duration.

## 1. Libraries

In [ ]:
# If any library is missing, remove the # at the start of the line below and run it:
# !pip install pandas numpy matplotlib statsmodels scipy scikit-learn python-docx openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut

from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.section import WD_ORIENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

import io
import os
import warnings
warnings.filterwarnings('ignore')  # MLPRegressor's lbfgs convergence warnings can appear excessively often

print("Libraries loaded.")

## 2. Settings and Constants

The parts that are SHARED with the Regression (MLR) notebook (group definitions, candidate variables, seed, Y-randomization repeat count, k-fold thresholds) were kept EXACTLY THE SAME - so that the two reports are directly comparable. New settings SPECIFIC to ANN (hyperparameter search range, inner-CV fold count, seed repeat count) are explained separately.

In [ ]:
# --- File path -------------------------------------------------------------
EXCEL_PATH = "Alcohol_Data.xlsx"

# --- Candidate descriptors ------------------------------------------------------
# WHY ALL OF THEM (no elimination): the reason VIF exists (the (X'X)^-1 matrix
# inverse in OLS becoming unstable under multicollinearity) does NOT APPLY to
# ANN - ANN weights are learned via gradient descent, there is no such matrix
# inversion step. That is why ALL 5 candidates are given to ANN, without elimination.
CANDIDATE_VARIABLES = ['Per(A)', 'A(2,1)', 'A(2,2)', 'tr(A)', 'epsilon(A)']

# --- The 6 independent descriptor groups (IDENTICAL to the MLR notebook) -----------
GROUP_SHEETS = {
    'CW-I (H-Suppressed)': 'CW-I_without_H',
    'CW-II (H-Suppressed)': 'CW-II_without_H',
    'CW-mix (H-Suppressed)': 'CW-mix_without_H',
    'CW-I (H-Included)': 'CW-I_with_H',
    'CW-II (H-Included)': 'CW-II_with_H',
    'CW-mix (H-Included)': 'CW-mix_with_H',
}

# --- The English form of group names as they will appear in plots (figures ONLY) --
GROUP_ENGLISH_NAME = {
    'CW-I (H-Suppressed)': 'CW-I (H-Suppressed)',
    'CW-II (H-Suppressed)': 'CW-II (H-Suppressed)',
    'CW-mix (H-Suppressed)': 'CW-mix (H-Suppressed)',
    'CW-I (H-Included)': 'CW-I (H-Included)',
    'CW-II (H-Included)': 'CW-II (H-Included)',
    'CW-mix (H-Included)': 'CW-mix (H-Included)',
}

# --- bp property (this notebook works ONLY for bp) -------------------------
BP_COLUMN = 'bp - Boiling Point (°C)'
BP_UNIT = '°C'
MOLECULE_COLUMN_PROPERTY = 'Aliphatic Alcohol'
MOLECULE_COLUMN_GROUP = 'Solvent'

# --- Fixed random seed (SAME value as MLR) -----------------------------------
RANDOM_SEED = 42

# --- Y-randomization repeat count (SAME as MLR) -----------------------------
YRAND_REPEATS = 1000

# --- k-fold thresholds (SAME as MLR) -------------------------------------------
K5_MIN_N = 15
K10_MIN_N = 30

# =============================================================================
# ANN-SPECIFIC SETTINGS (newly added parts with no counterpart in MLR)
# =============================================================================

# --- Hyperparameter search range (scanned in the INNER loop of nested CV) ------
# WHY THIS RANGE: in response to Reviewer 1's "why 4 neurons?" criticism,
# instead of picking a single architecture at random, we perform a SYSTEMATIC
# search. 1-2 hidden layers, 2/4/8/16 neurons per layer - a reasonable range
# for a data set as small as n=66 (larger architectures cannot be trained
# meaningfully with this little data).
HIDDEN_LAYER_OPTIONS = [
    (2,), (4,), (8,), (16,),
    (2, 2), (4, 4), (8, 8), (16, 16),
]
ALPHA_OPTIONS = [0.0001, 0.001, 0.01, 0.1]  # L2 regularization strength

# --- Fixed architecture settings ------------------------------------------------
ANN_ACTIVATION = 'relu'
# WHY lbfgs: scikit-learn's own documentation states that for small data sets
# (in our case n<=66 training samples) lbfgs converges faster and gives more
# stable results than adam/sgd - mini-batch-based methods offer no advantage
# with this little data.
ANN_SOLVER = 'lbfgs'
ANN_MAX_ITER = 2000

# --- Inner-CV fold count (for hyperparameter search) ---------------------------
INNER_KFOLD = 5

# --- FIXED seed list for the multi-seed (10-seed) repetition -------------------
# WHY: In item 13, Reviewer 1 criticized that "early stopping / convergence
# behavior was not reported reproducibly". Instead of giving a single
# "stopped at X iterations" number from one seed, we train with SEED_COUNT
# different random starts and report the mean±std iteration count (and this
# logic has now been generalized, consistently with the 10-seed mean±std
# principle requested for GCN in item 5, to EVERY place a neural network is
# trained - see Method Description, section 3).
#
# WHY THESE SPECIFIC 10 NUMBERS: these seeds are not random/made up on the
# spot - they were compiled from real code-based frequency analyses (a
# searchcode API scan, a scan of popular R packages on GitHub) of the most
# commonly encountered individual random-seed values: 0 and 1 (the most
# common "plain" choices), 1234 and 12345 (common sequential-digit choices),
# 42 (the most "iconic"/recognized individual seed in the ML community, via
# the Hitchhiker's Guide to the Galaxy reference), 100/123 (common
# round/plain numbers), 7 (a common "lucky number" choice), 2020/2021
# (common year-referenced choices). This list is used by taking the first
# SEED_COUNT elements - so even if SEED_COUNT is reduced for testing (e.g. in
# a scenario test), it is ALWAYS a prefix of this same fixed list; the
# formula does not change.
SEED_LIST = [0, 1, 7, 42, 100, 123, 1234, 2020, 2021, 12345]
SEED_COUNT = len(SEED_LIST)

print("Settings loaded. EXCEL_PATH =", EXCEL_PATH)

## 3. Data Loading

In [ ]:
def load_data(excel_path=EXCEL_PATH):
    """Loads all sheets in the Excel file and strips whitespace from molecule names."""
    properties_df = pd.read_excel(excel_path, sheet_name='All_Properties')
    properties_df[MOLECULE_COLUMN_PROPERTY] = properties_df[MOLECULE_COLUMN_PROPERTY].astype(str).str.strip()

    groups = {}
    for group_name, sheet_name in GROUP_SHEETS.items():
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        df[MOLECULE_COLUMN_GROUP] = df[MOLECULE_COLUMN_GROUP].astype(str).str.strip()
        groups[group_name] = df

    print(f"Loaded: {len(properties_df)} molecules, {len(groups)} descriptor groups.")
    return properties_df, groups


def prepare_bp_group_data(properties_df, groups, group_name):
    """Prepares the filtered X (5 descriptors, ALL of them) and Y (bp) arrays for
    bp, for a given group. Same logic (missing-data rules) as the same-named
    function in the MLR notebook - the difference is that there is NO excluded
    variable, ALL 5 descriptors are always used together (see the rationale in
    the Settings cell)."""
    property_series = pd.to_numeric(properties_df[BP_COLUMN], errors='coerce')
    valid_mask = property_series.notna()

    molecules = properties_df.loc[valid_mask, MOLECULE_COLUMN_PROPERTY].reset_index(drop=True)
    y = property_series.loc[valid_mask].reset_index(drop=True)

    group_df = groups[group_name].set_index(MOLECULE_COLUMN_GROUP)

    missing_molecules = [m for m in molecules if m not in group_df.index]
    if missing_molecules:
        print(f"WARNING: molecules not found in the {group_name} sheet are being skipped: {missing_molecules}")
        molecules = molecules[~molecules.isin(missing_molecules)].reset_index(drop=True)
        y = y[properties_df.loc[valid_mask, MOLECULE_COLUMN_PROPERTY].isin(molecules).values].reset_index(drop=True)

    x = group_df.loc[molecules.values, CANDIDATE_VARIABLES].reset_index(drop=True)
    x = x.apply(pd.to_numeric, errors='coerce')

    complete_mask = x.notna().all(axis=1)
    if not complete_mask.all():
        print(f"WARNING: {(~complete_mask).sum()} molecules have missing/non-numeric descriptor data, skipping.")
    x = x.loc[complete_mask].reset_index(drop=True)
    y = y.loc[complete_mask].reset_index(drop=True)
    molecules = molecules.loc[complete_mask].reset_index(drop=True)

    return molecules, x, y


def calculate_std_dev(y):
    return float(np.std(y.values, ddof=1))


def calculate_q2_and_s(y_actual, predictions):
    """General-purpose function that calculates R^2/Q^2 and S (standard error)
    for any (actual, prediction) pair - used repeatedly both in the MLR
    reference and in ANN's nested CV/Y-randomization results."""
    y_arr = np.asarray(y_actual, dtype=float)
    t_arr = np.asarray(predictions, dtype=float)
    residuals = y_arr - t_arr
    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((y_arr - y_arr.mean()) ** 2))
    q2 = 1 - ss_res / ss_tot
    s = float(np.sqrt(ss_res / len(y_arr)))
    return q2, s


def calculate_ccc(actual, prediction):
    """Concordance Correlation Coefficient (Lin, 1989) - EXACTLY THE SAME
    function as in the MLR notebook (see the Method and Terms Description in
    that notebook)."""
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    mean_actual, mean_prediction = float(np.mean(actual)), float(np.mean(prediction))
    var_actual, var_prediction = float(np.var(actual)), float(np.var(prediction))
    covariance = float(np.mean((actual - mean_actual) * (prediction - mean_prediction)))
    return (2 * covariance) / (var_actual + var_prediction + (mean_actual - mean_prediction) ** 2)


def calculate_applicability_domain(x, variables, y, outer_predictions, s_outer):
    """Leverage/standardized residual for the Williams Plot - EXACTLY THE SAME
    math as the same-named function in the MLR notebook (leverage depends
    ONLY on X, not on the model - so it can be used unchanged for ANN too).
    The difference: since ANN eliminates no variable, variables is ALWAYS
    CANDIDATE_VARIABLES (all 5), not a small post-VIF subset as in MLR."""
    X = sm.add_constant(x[variables])
    X_mat = X.values
    H = X_mat @ np.linalg.pinv(X_mat.T @ X_mat) @ X_mat.T
    leverage = np.diag(H)

    p_prime = len(variables) + 1
    n = len(y)
    h_star = 3 * p_prime / n

    standardized_residual = (y.values - outer_predictions) / s_outer if s_outer > 0 else np.zeros(n)

    outside_mask = (leverage > h_star) | (np.abs(standardized_residual) > 3)
    return leverage, standardized_residual, h_star, outside_mask

## 4. MLR Reference (For Comparison)

This cell re-runs, inside this notebook, the EXACT SAME VIF + p_max + OLS + LOOCV + k-fold + CCC process as in `MLR_Analysis.ipynb` - so that we can compare the ANN results with MLR FROM SCRATCH, using up-to-date data (instead of trying to read an old/stale Word file). VIF_THRESHOLD is the SAME as the MLR notebook (5).

In [ ]:
VIF_THRESHOLD = 5


def calculate_univariate_r2(x, y):
    r2_values = {}
    for variable in x.columns:
        X_single = sm.add_constant(x[[variable]])
        model = sm.OLS(y, X_single).fit()
        r2_values[variable] = model.rsquared
    return r2_values


def calculate_vif(x):
    X_with_const = sm.add_constant(x)
    return {
        column: variance_inflation_factor(X_with_const.values, i + 1)
        for i, column in enumerate(x.columns)
    }


def eliminate_variables_by_vif(x, y, threshold=VIF_THRESHOLD):
    remaining = list(x.columns)
    history = []
    while len(remaining) > 1:
        vif_values = calculate_vif(x[remaining])
        max_vif = max(vif_values.values())
        if max_vif <= threshold:
            break
        candidates = [k for k, v in vif_values.items() if abs(v - max_vif) < 0.01]
        if len(candidates) > 1:
            single_r2 = calculate_univariate_r2(x[candidates], y)
            to_remove = min(single_r2, key=single_r2.get)
        else:
            to_remove = candidates[0]
        history.append((to_remove, vif_values[to_remove], f'Eliminated (VIF>{threshold})'))
        remaining.remove(to_remove)
    return remaining, history


def determine_model_size(n, post_vif_variables, x, y):
    p_max = max(1, n // 5)
    selected = list(post_vif_variables)
    while len(selected) > p_max:
        single_r2 = calculate_univariate_r2(x[selected], y)
        weakest = min(single_r2, key=single_r2.get)
        selected.remove(weakest)
    return selected, p_max


def build_model(x, y, variables):
    X = sm.add_constant(x[variables])
    return sm.OLS(y, X).fit()


def apply_loocv(x, y, variables):
    n = len(y)
    predictions = np.zeros(n)
    for i in range(n):
        train_idx = [j for j in range(n) if j != i]
        X_train = sm.add_constant(x.iloc[train_idx][variables], has_constant='add')
        model = sm.OLS(y.iloc[train_idx], X_train).fit()
        X_test = sm.add_constant(x.iloc[[i]][variables], has_constant='add')
        X_test = X_test.reindex(columns=X_train.columns, fill_value=1.0)
        predictions[i] = model.predict(X_test).values[0]
    q2_loo, s_loo = calculate_q2_and_s(y, predictions)
    return q2_loo, s_loo, predictions


def apply_kfold_cv(x, y, variables, k, seed=RANDOM_SEED):
    n = len(y)
    rng = np.random.default_rng(seed)
    indices = np.arange(n)
    rng.shuffle(indices)
    folds = np.array_split(indices, k)
    predictions = np.zeros(n)
    for fold in folds:
        test_idx = fold
        train_idx = np.setdiff1d(indices, test_idx)
        X_train = sm.add_constant(x.iloc[train_idx][variables], has_constant='add')
        model = sm.OLS(y.iloc[train_idx], X_train).fit()
        X_test = sm.add_constant(x.iloc[test_idx][variables], has_constant='add')
        X_test = X_test.reindex(columns=X_train.columns, fill_value=1.0)
        predictions[test_idx] = model.predict(X_test).values
    q2_k, s_k = calculate_q2_and_s(y, predictions)
    return q2_k, s_k


def calculate_mlr_reference(x, y):
    """Calculates the result MLR would give for bp in this group (from
    scratch, with up-to-date data) - used in the ANN comparison table.
    s_fit/s_loo/s_k5/s_k10 are RMSE values computed DIRECTLY from the
    corresponding prediction set (sqrt(SS_residual/n)) - they are NOT
    derived INDIRECTLY from sd (the (n-1)-based SD) in the report tables."""
    n = len(y)
    vif_remaining, _ = eliminate_variables_by_vif(x, y)
    selected_variables, p_max = determine_model_size(n, vif_remaining, x, y)
    model = build_model(x, y, selected_variables)
    r2_fit = model.rsquared
    s_fit = float(np.sqrt(model.ssr / n))
    q2_loo, s_loo, loocv_predictions = apply_loocv(x, y, selected_variables)
    q2_k5 = s_k5 = q2_k10 = s_k10 = None
    if n >= K5_MIN_N:
        q2_k5, s_k5 = apply_kfold_cv(x, y, selected_variables, k=5)
    if n >= K10_MIN_N:
        q2_k10, s_k10 = apply_kfold_cv(x, y, selected_variables, k=10)
    ccc = calculate_ccc(y.values, loocv_predictions)
    return {
        'variables': selected_variables, 'p_max': p_max,
        'r2_fit': r2_fit, 's_fit': s_fit,
        'q2_loo': q2_loo, 's_loo': s_loo,
        'q2_k5': q2_k5, 's_k5': s_k5,
        'q2_k10': q2_k10, 's_k10': s_k10, 'ccc': ccc,
    }

## 5. ANN — Nested Cross-Validation, Hyperparameter Search, and Multi-Seed Repetitions

This cell builds the core validation engine for ANN: in every outer-CV round (each molecule of LOOCV, or each fold of k-fold), the hyperparameter search is performed FROM SCRATCH and is LIMITED ONLY to that round's training data - the held-out test molecule/fold is never looked at at any stage. In addition, to address the concern that a result obtained with a SINGLE seed could turn out lucky/unlucky due to the randomness of the neural network's weight initialization (see the Method Description section, a generalization of the logic in Reviewer 1 item 13), the entire nested-CV process is repeated FROM SCRATCH with SEED_COUNT (10) different main seeds, and the results are accumulated as mean±std.

In [ ]:
from collections import defaultdict


def build_ann_base_pipeline(seed):
    """Scaler (StandardScaler) + MLPRegressor - together inside a Pipeline, so
    that during cross-validation the scaler is fit EACH TIME ONLY on that
    round's training data (there is NO leakage whatsoever into the test/outer
    data - sklearn's Pipeline guarantees this automatically). The MLP's
    random_state depends on the seed passed in from outside - so the weight
    initialization differs for every (main seed, outer round) combination."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(activation=ANN_ACTIVATION, solver=ANN_SOLVER,
                              max_iter=ANN_MAX_ITER, random_state=seed)),
    ])


def build_inner_search(seed):
    """Inner-CV (hyperparameter search) - a GridSearchCV with INNER_KFOLD folds,
    shuffled according to the given seed (both the inner-CV split and the MLP
    weight initialization depend on the SAME seed - a single seed determines
    everything about that round)."""
    param_grid = {
        'mlp__hidden_layer_sizes': HIDDEN_LAYER_OPTIONS,
        'mlp__alpha': ALPHA_OPTIONS,
    }
    inner_cv = KFold(n_splits=INNER_KFOLD, shuffle=True, random_state=seed)
    return GridSearchCV(build_ann_base_pipeline(seed), param_grid, cv=inner_cv,
                         scoring='r2', n_jobs=-1)


def nested_cross_validation_single_seed(x, y, outer_splitter, main_seed):
    """
    The nested CV run from start to finish with a SINGLE 'main seed'.
    outer_splitter can be sklearn's LeaveOneOut() or a KFold(k, shuffle=True,
    random_state=...) object. Each outer round's own inner search runs with a
    UNIQUE seed derived from the (main_seed, round_idx) pair - so even within
    the same main_seed, rounds use different inner-CV splits/weight
    initializations, but from a VERY DIFFERENT starting point with different
    main_seed values.

    CHANGE (mean-score instead of vote-counting): in every outer round, the
    average inner-CV score (mean_test_score) of ALL 32 combinations tried by
    the inner search is recorded - not just that round's winner. The reason:
    in cases where the margin of victory among the 32 combinations is very
    small (e.g. 48 votes to 47 votes), simply counting 'who won' made the
    selection overly sensitive to noise - a combination could be declared the
    'winner' just by being lucky in a few rounds, even if it performed poorly
    in other rounds.
    """
    X_arr = x[CANDIDATE_VARIABLES].values
    y_arr = y.values
    n = len(y_arr)
    predictions = np.zeros(n)
    all_scores = []  # [((architecture, alpha), score), ...] - EVERY combination of EVERY round

    for round_idx, (train_idx, test_idx) in enumerate(outer_splitter.split(X_arr)):
        round_seed = main_seed * 10_000 + round_idx
        search = build_inner_search(seed=round_seed)
        search.fit(X_arr[train_idx], y_arr[train_idx])
        predictions[test_idx] = search.predict(X_arr[test_idx])

        # GridSearchCV already keeps the inner-CV score of all 32 combinations
        # in cv_results_ - we record this information too (not just the
        # winner, but all of it) without needing any extra model training.
        for params, score in zip(search.cv_results_['params'], search.cv_results_['mean_test_score']):
            key = (params['mlp__hidden_layer_sizes'], params['mlp__alpha'])
            all_scores.append((key, score))

    return predictions, all_scores


def _mean_std(values):
    if len(values) == 0:
        return None, None
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
    return mean, std


def nested_cross_validation_multi_seed(x, y, outer_splitter_factory, seed_count=SEED_COUNT):
    """
    Repeats nested_cross_validation_single_seed() from start to finish with
    SEED_COUNT different main seeds. outer_splitter_factory is a function that
    takes a seed and returns an outer splitter object specific to that
    repetition (for LOOCV the seed does not matter/the same 66 rounds are
    always formed; for k-fold, a DIFFERENT split is formed depending on the
    seed, so each of the 10 repetitions also tries a different fold
    distribution).

    Returned dict:
      - q2_mean / q2_std / q2_list: the mean/std/full list of the Q2 computed
        from each repetition's OWN outer predictions.
      - s_mean / s_std / s_list: likewise, for the RMSE (S = sqrt(SS_residual/n))
        computed DIRECTLY from EACH repetition's own outer predictions - not
        derived INDIRECTLY from sd (the (n-1)-based SD) in the report tables.
      - ccc_mean / ccc_std / ccc_list: likewise for CCC.
      - mean_prediction: for each molecule, the MEAN of the outer predictions
        over the 10 repetitions - used for the Williams Plot / Applicability
        Domain.
      - scores_pool: the COMBINED (pooled) list of the scores of ALL 32
        combinations from ALL outer rounds of the 10 repetitions - the final
        architecture is now selected based on the MEAN score in this pool.
    """
    q2_list = []
    s_list = []
    ccc_list = []
    prediction_matrix = []
    all_scores_pool = []

    for i in range(seed_count):
        main_seed = SEED_LIST[i]
        splitter = outer_splitter_factory(main_seed)
        predictions, scores = nested_cross_validation_single_seed(x, y, splitter, main_seed)
        q2, s = calculate_q2_and_s(y, predictions)
        ccc = calculate_ccc(y.values, predictions)
        q2_list.append(q2)
        s_list.append(s)
        ccc_list.append(ccc)
        prediction_matrix.append(predictions)
        all_scores_pool.extend(scores)

    prediction_matrix = np.array(prediction_matrix)
    mean_prediction = prediction_matrix.mean(axis=0)

    q2_mean, q2_std = _mean_std(q2_list)
    s_mean, s_std = _mean_std(s_list)
    ccc_mean, ccc_std = _mean_std(ccc_list)

    return {
        'q2_mean': q2_mean, 'q2_std': q2_std, 'q2_list': q2_list,
        's_mean': s_mean, 's_std': s_std, 's_list': s_list,
        'ccc_mean': ccc_mean, 'ccc_std': ccc_std, 'ccc_list': ccc_list,
        'mean_prediction': mean_prediction,
        'scores_pool': all_scores_pool,
        'repeat_count': seed_count,
    }


def rank_by_mean_score(scores_pool):
    """Calculates the MEAN inner-CV score of each (architecture, alpha)
    combination across ALL outer-round x seed evaluations (e.g. 66x10=660
    values for bp LOOCV) and sorts them from largest to smallest. The first
    element [0] of the returned list is used as the final architecture - a
    choice based on 'how well it performed on average overall', rather than
    'how many times it came first'."""
    score_totals = defaultdict(list)
    for key, score in scores_pool:
        score_totals[key].append(score)
    means = {k: float(np.mean(v)) for k, v in score_totals.items()}
    return sorted(means.items(), key=lambda kv: kv[1], reverse=True)


def train_and_report_final_model(x, y, architecture, alpha, seed_count=SEED_COUNT):
    """
    Trains the final/official model using the 'winning' architecture selected
    from nested CV, on ALL the data (all molecules in this group). It is
    trained repeatedly with SEED_COUNT different random starts; both how many
    iterations it took to converge (Reviewer 1 item 13) and R2_fit (for
    consistency, the same multi-seed standard) are reported as mean±std.
    """
    X_arr = x[CANDIDATE_VARIABLES].values
    y_arr = y.values
    iterations = []
    r2_list = []
    s_fit_list = []
    models = []
    for i in range(seed_count):
        seed = SEED_LIST[i]
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', MLPRegressor(hidden_layer_sizes=architecture, alpha=alpha,
                                  activation=ANN_ACTIVATION, solver=ANN_SOLVER,
                                  max_iter=ANN_MAX_ITER, random_state=seed)),
        ])
        pipeline.fit(X_arr, y_arr)
        iterations.append(int(pipeline.named_steps['mlp'].n_iter_))
        prediction = pipeline.predict(X_arr)
        r2_i, s_i = calculate_q2_and_s(y, prediction)
        r2_list.append(r2_i)
        s_fit_list.append(s_i)
        models.append(pipeline)

    mean_iter, std_iter = _mean_std(iterations)
    r2_mean, r2_std = _mean_std(r2_list)
    s_fit_mean, s_fit_std = _mean_std(s_fit_list)

    # WHY THE SAME (FIRST) MODEL IS ALWAYS KEPT: if a single, reproducible
    # model object is needed for visualization/further use, models[0] is kept
    # fixed so that it is always the model trained with the same seed
    # (SEED_LIST[0]) - but the reported R2_fit is no longer THIS one, it is
    # the MEAN of the 10 seeds.
    final_model = models[0]

    return {
        'model': final_model, 'architecture': architecture, 'alpha': alpha,
        'mean_iteration': mean_iter, 'std_iteration': std_iter,
        'all_iterations': iterations,
        'r2_fit_mean': r2_mean, 'r2_fit_std': r2_std, 'r2_list': r2_list,
        's_fit_mean': s_fit_mean, 's_fit_std': s_fit_std, 's_fit_list': s_fit_list,
    }


def apply_y_randomization_ann(x, y, architecture, alpha, actual_r2_list,
                               repeats=YRAND_REPEATS, seed_count=SEED_COUNT):
    """
    SAME logic as apply_y_randomization in MLR: with a FIXED architecture
    (the winner of nested CV), the R^2 of the actual model fit (in-sample) on
    ALL the data is compared against the R^2 distribution of models in which Y
    was randomly shuffled, repeatedly (1000 times).

    IMPORTANT FOR CONSISTENCY: both the actual model AND EACH randomized
    Y-shuffle trial are evaluated by the SAME standard (trained with
    seed_count different seeds and averaged) - so that we do not clean up the
    'actual' side with a 10-seed mean while leaving the 'random' side dirty
    with a single seed; both sides are compared at the same noise level. The
    actual_r2_list parameter is the 10-seed r2_list ALREADY computed by
    train_and_report_final_model() - it is taken from outside so the same
    computation is not repeated.
    """
    X_arr = x[CANDIDATE_VARIABLES].values
    y_arr = y.values.copy()
    rng = np.random.default_rng(RANDOM_SEED)

    def _multi_seed_r2(y_target, seed_base):
        r2_values = []
        for i in range(seed_count):
            seed = seed_base + SEED_LIST[i]
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('mlp', MLPRegressor(hidden_layer_sizes=architecture, alpha=alpha,
                                      activation=ANN_ACTIVATION, solver=ANN_SOLVER,
                                      max_iter=ANN_MAX_ITER, random_state=seed)),
            ])
            pipeline.fit(X_arr, y_target)
            r2_i, _ = calculate_q2_and_s(y_target, pipeline.predict(X_arr))
            r2_values.append(r2_i)
        return float(np.mean(r2_values))

    actual_r2 = float(np.mean(actual_r2_list))

    random_r2s = np.zeros(repeats)
    for i in range(repeats):
        y_shuffled = rng.permutation(y_arr)
        # each permutation has its own base - separated by a step many times
        # larger than even the largest element of SEED_LIST (12345), so that
        # no permutation trial (or the actual model's SEED_LIST seeds) ever
        # collides - yet the SAME 10 SEED_LIST values are still added on top
        # of each base to compute the 10-seed mean (for consistency)
        random_r2s[i] = _multi_seed_r2(y_shuffled, seed_base=(i + 1) * 1_000_000)

    return actual_r2, random_r2s

## 6. Word Report — Helper Functions

EXACTLY THE SAME as the same-named functions in `MLR_Analysis.ipynb` (table/cell formatting is language-independent, unchanged).

In [ ]:
NAVY = RGBColor(0x1F, 0x38, 0x64)
ACCENT = RGBColor(0x44, 0x72, 0xC4)
GRAY = RGBColor(0x7F, 0x7F, 0x7F)
RED = RGBColor(0xC0, 0x39, 0x2B)


def _set_cell_background(cell, hex_color):
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:fill'), hex_color)
    cell._tc.get_or_add_tcPr().append(shd)


def write_cell(cell, lines, bold_first=True, header=False, gray_subsequent=True, align=WD_ALIGN_PARAGRAPH.CENTER):
    if isinstance(lines, str):
        lines = [lines]
    cell.text = ''
    p = cell.paragraphs[0]
    p.alignment = align
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    for i, line in enumerate(lines):
        if i > 0:
            p.add_run().add_break(WD_BREAK.LINE)
        run = p.add_run(str(line))
        run.font.size = Pt(10 if i == 0 else 8)
        run.font.bold = header or (bold_first and i == 0)
        if header:
            run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        elif i > 0 and gray_subsequent:
            run.font.color.rgb = GRAY
    if header:
        _set_cell_background(cell, '4472C4')


def build_table(doc, headers, rows, widths=None):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Table Grid'
    for i, header in enumerate(headers):
        write_cell(table.rows[0].cells[i], header, header=True)
    for row in rows:
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            write_cell(row_cells[i], value)
    if widths:
        for row in table.rows:
            for i, w in enumerate(widths):
                row.cells[i].width = Inches(w)
    return table


def s_ratio_lines(value, s, unit):
    """Produces a two-line cell containing the R^2/Q^2 value, with its own
    RMSE (S = sqrt(SS_residual/n), in the original unit) and the ratio (%) of
    this S to Y's natural standard deviation (SD) underneath it. S is given
    DIRECTLY by the caller - it is NOT derived INDIRECTLY from sd (the
    (n-1)-based SD) (this used to be done that way, which did NOT MATCH the
    definition of RMSE)."""
    if value is None:
        return ['—']
    percent = round(np.sqrt(max(0.0, 1 - value)) * 100)
    return [f"{value:.3f}", f"S={s:.2f}{unit} (%{percent})"]


def s_ratio_lines_mean(mean, std, s, unit):
    """SAME visual format as s_ratio_lines, but the value is now the
    mean±std of SEED_COUNT different seeds. S is likewise the mean of the OWN
    RMSE values of SEED_COUNT different seeds (DIRECTLY, not INDIRECTLY from
    sd)."""
    if mean is None:
        return ['—']
    percent = round(np.sqrt(max(0.0, 1 - mean)) * 100)
    return [f"{mean:.3f} ± {std:.3f}", f"S={s:.2f}{unit} (%{percent})"]


def add_heading(doc, text, level=1, color=NAVY):
    p = doc.add_heading(level=level)
    run = p.add_run(text)
    run.font.color.rgb = color
    return p


def add_paragraph(doc, text, italic=False, bold=False, color=None, size=11):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.font.italic = italic
    run.font.bold = bold
    if color:
        run.font.color.rgb = color
    return p

## 7. Figures

The same structure of the plotting functions in `MLR_Analysis.ipynb` (dashed threshold lines, region labels) is preserved - only '(ANN)' was added to the titles, so they are not confused with the same figures in the MLR report.

In [ ]:
def plot_yrand_figure(actual_r2, random_r2s, title):
    fig, ax = plt.subplots(figsize=(6.2, 3.6), dpi=150)
    ax.hist(random_r2s, bins=30, color='#9DB8D9', edgecolor='#4472C4',
            alpha=0.85, label=f'Randomly permuted Y ({YRAND_REPEATS} trials)')
    ax.axvline(actual_r2, color='#C0392B', linewidth=2.2,
               label=f"Actual model (R²={actual_r2:.3f})")
    ax.set_xlabel('R² value')
    ax.set_ylabel('Number of trials (frequency)')
    ax.set_title(f'Y-Randomization Test (ANN) — {title}', fontsize=10.5)
    ax.set_xlim(min(0, float(np.min(random_r2s)) - 0.05), 1.0)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, linestyle='--')
    fig.tight_layout()
    return fig


def plot_williams_plot(leverage, standardized_residual, h_star, molecule_names, title):
    fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=150)
    outside_mask = (leverage > h_star) | (np.abs(standardized_residual) > 3)

    ax.scatter(leverage[~outside_mask], standardized_residual[~outside_mask], s=26,
               color='#4472C4', alpha=0.75, edgecolor='white', linewidth=0.4,
               label='Training set molecules')
    ax.scatter(leverage[outside_mask], standardized_residual[outside_mask], s=55,
               color='#C0392B', marker='D', edgecolor='black', linewidth=0.5,
               label='Molecules outside AD', zorder=5)

    # Same logic as plot_williams_plot in the MLR notebook: the name of every
    # molecule outside the AD is written as a small reddish label right next
    # to its point - this was done in MLR, was missing in ANN, and is
    # completed here.
    outside_idx = np.where(outside_mask)[0]
    for i, molecule in zip(outside_idx, molecule_names):
        ax.annotate(str(molecule), (leverage[i], standardized_residual[i]), textcoords='offset points',
                    xytext=(6, 4), fontsize=7.5, color='#7B241C')

    ax.axhline(3, color='gray', linestyle='--', linewidth=1)
    ax.axhline(-3, color='gray', linestyle='--', linewidth=1)
    ax.axvline(h_star, color='gray', linestyle='--', linewidth=1)

    label_style = dict(fontsize=7, color='#888888', alpha=0.9, style='italic',
                        transform=ax.transAxes)
    ax.text(0.03, 0.95, 'Y-outlier', ha='left', va='top', **label_style)
    ax.text(0.03, 0.05, 'Y-outlier', ha='left', va='bottom', **label_style)
    ax.text(0.97, 0.95, 'High-risk', ha='right', va='top', **label_style)
    ax.text(0.97, 0.05, 'High-risk', ha='right', va='bottom', **label_style)
    ax.text(0.97, 0.5, 'Good leverage', ha='right', va='center', **label_style)

    ax.set_xlabel('Leverage (h)')
    ax.set_ylabel('Standardized Residual')
    ax.set_title(f'Williams Plot — Applicability Domain (ANN) — {title}', fontsize=10.5)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, linestyle='--')
    fig.tight_layout()
    return fig


def fig_to_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    plt.close(fig)
    return buf

## 8. Method and Terms Description

A section, written ONCE per report, explaining every ANN-specific method/term used (what - how - why). SAME format/level of detail as the same section in the MLR notebook (What / How / Why like this).

In [ ]:
def write_method_and_terms_section(doc):
    add_heading(doc, "Method and Terms Description", level=1)

    add_heading(doc, "1. Why bp only", level=2)
    add_paragraph(doc, "What: the ANN (Artificial Neural Network) analysis is applied not to "
                        "all 12 properties, but ONLY to the bp (boiling point) property.")
    add_paragraph(doc, "How: bp is the MOST COMPLETE property in the data set - n=66, no "
                        "missing data at all. For most of the other properties, n is much "
                        "lower (e.g. thermal conductivity n=13).")
    add_paragraph(doc, "Why this way: ANN is a method that learns far more parameters than "
                        "MLR - training and validating it meaningfully on low-n properties "
                        "(such as n=13-14) would be nearly impossible (this was also "
                        "explicitly stated in the reviewer reports: generalization claims "
                        "for ANN/XGBoost/GCN on a small data set are not reliable). Focusing "
                        "on bp was chosen because it is the ONLY property for which we can "
                        "apply ANN properly, with a robust validation framework - moreover, "
                        "in the earlier (rejected) manuscript, ANN was already applied only "
                        "to bp.")

    add_heading(doc, "2. Input: all 5 descriptors, without elimination + normalization", level=2)
    add_paragraph(doc, "What: ANN is given, as input, ALL 5 descriptors (Per(A), A(2,1), "
                        "A(2,2), tr(A), epsilon(A)) rather than a VIF-eliminated subset as "
                        "in MLR.")
    add_paragraph(doc, "How: the reason VIF exists (the (X'X)^-1 matrix inverse in OLS "
                        "becoming unstable under multicollinearity) does not apply to ANN - "
                        "ANN weights are learned via gradient descent (backpropagation), "
                        "there is no such matrix inversion step. Before being given to ANN, "
                        "the inputs are standardized with StandardScaler (mean=0, std=1) - "
                        "this is a MANDATORY preprocessing step for ANN (unlike OLS, ANN is "
                        "sensitive to the scale of the inputs: gradient descent can converge "
                        "slowly/unstably with differently scaled inputs).")
    add_paragraph(doc, "Why this way: (1) VIF's mathematical rationale does not apply to "
                        "ANN, (2) the reviewer reports did not ask for variable elimination "
                        "for ML models, (3) with n=66 and 5 inputs the model is already "
                        "small, there is no need to eliminate. Standardization is applied "
                        "INSIDE an sklearn Pipeline (StandardScaler + MLPRegressor "
                        "together) - this way, in every cross-validation round the scaler "
                        "is fit ONLY on that round's training data, and there is no leakage "
                        "whatsoever into the test/outer data.")

    add_heading(doc, "3. Multi-Seed (10-Seed) Repetition - General Principle", level=2)
    add_paragraph(doc, f"What: everywhere in this report that a neural network is TRAINED "
                        f"(nested CV itself, the final model, each trial of "
                        f"Y-randomization), the process is repeated FROM SCRATCH not with a "
                        f"SINGLE random seed, but with {SEED_COUNT} DIFFERENT random seeds, "
                        f"and the results are reported as mean±std.")
    add_paragraph(doc, f"How: the neural network's weight initialization is a random "
                        f"process; in addition, in the k-fold validations (Q2(k=5), "
                        f"Q2(k=10)) the splitting into folds is also random. These two "
                        f"sources of randomness add a 'luck factor' that is INDEPENDENT of "
                        f"the data set itself - the same data, with the same method, can "
                        f"give a somewhat different result even when run with a different "
                        f"seed. To measure and report this, every stage is repeated with "
                        f"{SEED_COUNT} different main seeds; EVERY number that appears in "
                        f"the report (Q2LOO, Q2(k=5), Q2(k=10), CCC, R2_fit, iteration "
                        f"count, Y-randomization R2 values) is the mean±std of these "
                        f"{SEED_COUNT} repetitions - not the result of a single "
                        f"'lucky/unlucky' run.")
    add_paragraph(doc, f"The {SEED_COUNT} seeds used are FIXED and deterministic (the "
                        f"report gives the EXACT SAME result every time it is run): "
                        f"{SEED_LIST[:SEED_COUNT]}. These values were not made up at "
                        f"random/on the spot - they were compiled from real code-based "
                        f"frequency analyses (a searchcode API scan, a scan of popular R "
                        f"packages on GitHub) of the most commonly encountered individual "
                        f"random-seed values: 0 and 1 (the most common plain choices), 1234 "
                        f"and 12345 (common sequential-digit choices), 42 (the most "
                        f"recognized individual seed in the ML community, via the "
                        f"Hitchhiker's Guide to the Galaxy reference), 100/123 (common "
                        f"plain numbers), 7 (a common 'lucky number'), 2020/2021 (common "
                        f"year-referenced choices).")
    add_paragraph(doc, "Why this way: in item 5, Reviewer 1 explicitly requested, for GCN, "
                        "'using cross-validation repeated 10 times with different random "
                        "seeds... and reporting the mean ± standard deviation of R and "
                        "RMSE', and in item 13 requested that we report, for ANN, 'the "
                        "typical number of epochs (mean ± std across runs)'. The shared "
                        "logic of these two requests is the same: in a "
                        "neural-network-based method, a result obtained with a SINGLE seed "
                        "can turn out well or poorly by chance. There is no categorical "
                        "difference between GCN and ANN in this respect (both are neural "
                        "networks with random weight initialization) - therefore the "
                        "10-seed mean±std principle explicitly requested by Reviewer 1 for "
                        "GCN has been generalized, for consistency, to EVERY "
                        "neural-network-training step of ANN; the 10-seed reporting "
                        "requested in item 13 (for the iteration count alone) is thus "
                        "placed within a broader, consistent framework as well.")

    add_heading(doc, "4. Nested Cross-Validation", level=2)
    add_paragraph(doc, "What: a two-layer validation scheme used to produce ANN's "
                        "Q2LOO/Q2(k=5)/Q2(k=10)-equivalent results.")
    add_paragraph(doc, "How: the OUTER loop (e.g. each molecule in turn, for LOOCV) sets "
                        "aside one molecule/fold entirely. INSIDE the remaining data (the "
                        "outer-training set), a second cross-validation (the INNER loop, 5 "
                        "folds) begins: for every hyperparameter combination to be tried "
                        "(number of hidden layers/number of neurons x regularization "
                        "strength), it is trained and tested separately on the 5 "
                        "inner-loop folds, and its average performance is calculated. The "
                        "combination with the best average WINS. A model is then trained "
                        "using these winning settings on the ENTIRE outer-training set and "
                        "tested on the held-out, never-seen molecule/fold waiting OUTSIDE. "
                        "This is repeated for every molecule/fold in the outer loop; in the "
                        "end, a prediction set emerges in which EVERYONE has been used "
                        "exactly once as a 'never-seen test'.")
    add_paragraph(doc, f"In addition (see section 3): this entire nested process is "
                        f"repeated FROM SCRATCH with {SEED_COUNT} different main seeds - "
                        f"in every repetition, both the outer loop's k-fold shuffling "
                        f"(this does not apply to LOOCV, since in LOOCV a 'fold' is a "
                        f"single molecule and is not sensitive to shuffling) and the inner "
                        f"search's k-fold shuffling and weight initialization are done with "
                        f"a different seed. The Q2LOO/Q2(k=5)/Q2(k=10)/CCC values shown in "
                        f"the report are the mean±std of these {SEED_COUNT} repetitions.")
    add_paragraph(doc, "Why this way: if we were to perform both the hyperparameter "
                        "selection AND the final performance measurement on the SAME test "
                        "data, the result would come out artificially good (we would have "
                        "selected the best settings for that specific data, not true "
                        "generalizability) - it would be a fallacy similar to trying 9 "
                        "different (actually equally strong) hyperparameter combinations "
                        "and declaring whichever one happens to do best on that test data "
                        "the 'best architecture'. Nested CV prevents this fallacy by "
                        "statistically separating the question 'which setting is best' "
                        "from the question 'how good is it really' - a direct answer to "
                        "Reviewer 1's criticism that 'hyperparameters were "
                        "unjustified/not searched'.")

    add_heading(doc, "5. Hyperparameter search range and performance ranking", level=2)
    add_paragraph(doc, f"What: the hyperparameter combinations tried in the inner loop: "
                        f"number of hidden layers 1-2, neurons per layer {{2, 4, 8, 16}}, "
                        f"regularization (alpha, L2) {{0.0001, 0.001, 0.01, 0.1}} - a total "
                        f"of {len(HIDDEN_LAYER_OPTIONS) * len(ALPHA_OPTIONS)} combinations.")
    add_paragraph(doc, f"How: in EVERY round of the outer loop, the average inner-CV score "
                        f"(mean_test_score) of ALL 32 combinations already computed by the "
                        f"inner search (GridSearchCV) is recorded - not just that round's "
                        f"winner. Since this entire process is repeated with {SEED_COUNT} "
                        f"different seeds (see section 3), the evaluation pool consists of "
                        f"a total of (number of outer rounds x {SEED_COUNT}) evaluations. "
                        f"The MEAN of ALL scores in this pool is computed for every "
                        f"combination; the combination with the highest mean score is "
                        f"selected as the architecture of the 'final' model used in the "
                        f"report.")
    add_paragraph(doc, "Why this way: simply counting 'who came first how many times' "
                        "makes the selection overly sensitive to noise in cases where the "
                        "margin of victory among the 32 combinations is very small (e.g. a "
                        "single-digit vote difference between the winner and the runner-up "
                        "in some groups) - a combination could be declared the 'winner' "
                        "just by being lucky in a few rounds, even if it performed poorly "
                        "in other rounds. Taking the MEAN of all the evaluations measures "
                        "which combination performed best/most consistently OVERALL, and "
                        "removes the one-off luck factor - no extra model training is "
                        "needed, because these scores have already been computed by "
                        "GridSearchCV. This table provides concrete evidence against "
                        "Reviewer 1's 'why 4 neurons?' criticism - the architecture was not "
                        "chosen at random, it was searched systematically and based on "
                        "average performance.")

    add_heading(doc, "6. Final model and iteration/seed reporting", level=2)
    add_paragraph(doc, "What: the final/official model, trained using ALL molecules in "
                        "this group, with the winning architecture selected from nested "
                        "CV.")
    add_paragraph(doc, f"How: this model is trained repeatedly with {SEED_COUNT} different "
                        f"random starting seeds; each time, both how many iterations it "
                        f"took to converge (the lbfgs solver uses the concept of "
                        f"'iteration', not 'epoch') and the resulting R2_fit value are "
                        f"recorded; both are reported as mean±std. The R2_fit in the "
                        f"report is thus presented at the SAME standard (10-seed "
                        f"mean±std) as Q2LOO and the others.")
    add_paragraph(doc, "Why this way: in item 13, Reviewer 1 criticized that 'early "
                        "stopping/convergence behavior was not reported reproducibly' "
                        "(the earlier ANN had a vague statement like 'generally stopped "
                        "earlier'). Retraining with multiple seeds and reporting mean±std "
                        "gives a concrete and reproducible answer to this criticism; "
                        "reporting R2_fit at the same standard shows that the final model "
                        "rests on consistent behavior, not on a single lucky seed.")

    add_heading(doc, "7. Y-Randomization (adapted to ANN)", level=2)
    add_paragraph(doc, "What: the SAME logic as Y-randomization in MLR - the Y values are "
                        "shuffled randomly (breaking the molecule-target pairing), and a "
                        "model is built repeatedly (1000 times) with a FIXED architecture "
                        "(the winner of nested CV).")
    add_paragraph(doc, f"How: the actual model's R2 (on all the data, in-sample) is "
                        f"compared against the R2 distribution of 1000 random "
                        f"Y-shuffling trials. IMPORTANT: we do NOT repeat nested CV 1000 "
                        f"times here (this would take hours computationally) - MLR's own "
                        f"Y-randomization already uses in-sample R2_fit, not LOOCV. "
                        f"Applying the same method to ANN provides an EXACTLY THE "
                        f"SAME/consistent comparison with MLR - this is not a 'shortcut', "
                        f"it is a faithful adaptation to ANN of the method MLR already "
                        f"uses. In addition (see section 3), the actual model AND EACH of "
                        f"the 1000 random trials are computed, against the randomness of "
                        f"weight initialization, at the same standard, as the mean of "
                        f"{SEED_COUNT} different seeds - so that both sides of the "
                        f"Y-randomization comparison (the actual model and the random "
                        f"distribution) are equally cleaned of 'weight-initialization "
                        f"noise', and no one-sided advantage/disadvantage arises.")
    add_paragraph(doc, "Why this way: it directly and empirically answers the question "
                        "posed by the classical F/P test - 'is this relationship based on "
                        "chance?' - without relying on distributional assumptions. The "
                        "interpretation of the result is THREE-TIERED, as in MLR: if the "
                        "actual R2 is above the great majority of the random "
                        "distribution, this is STRONG support; if it is above the mean "
                        "but not clearly separated, it should be interpreted CAUTIOUSLY; "
                        "if it is at or below the mean, it is considered INVALID.")

    add_heading(doc, "8. CCC (Concordance Correlation Coefficient)", level=2)
    add_paragraph(doc, "Explained in detail in the MLR notebook (Lin, 1989 - a "
                        "concordance coefficient that jointly measures correlation "
                        "STRENGTH and CLOSENESS to the true y=x line). Here it is "
                        "computed, with the EXACT SAME formula, from ANN's out-of-fold "
                        "predictions from nested LOOCV; these predictions themselves (see "
                        "sections 3-4) are already the mean of 10 seeds, so CCC too is "
                        "reported as a 10-seed mean±std.")

    add_heading(doc, "9. Applicability Domain / Williams Plot (adapted to ANN)", level=2)
    add_paragraph(doc, "What: the Williams Plot logic explained in detail in the MLR "
                        "notebook (leverage h, threshold h*=3p'/n; standardized residual, "
                        "threshold ±3; four regions: normal / well-behaved extreme point / "
                        "Y-outlier / high-risk).")
    add_paragraph(doc, "How: leverage MATHEMATICALLY depends ONLY on the descriptor (X) "
                        "values, not on which model (OLS or ANN) is used - so the same "
                        "formula h=X(X'X)^-1X' applies to ANN unchanged. The only "
                        "difference: here p'=6 is used (ALL 5 descriptors + a column "
                        "equivalent to the constant term) - not a small post-VIF subset as "
                        "in MLR, because ANN is given all variables without any "
                        "elimination (see item 2). The standardized residual, in turn, is "
                        "computed from ANN's own out-of-fold predictions from nested "
                        "LOOCV.")
    add_paragraph(doc, "Why this way: leverage not depending on the model lets us perform "
                        "a consistent/comparable Applicability Domain analysis between "
                        "MLR and ANN - the only difference is how many descriptors are "
                        "used (5 vs. fewer after VIF) and which model the residuals come "
                        "from.")

    add_heading(doc, "10. How to read the MLR vs ANN comparison", level=2)
    add_paragraph(doc, "In the comparison table below, the MLR results have been "
                        "recomputed FROM SCRATCH, with up-to-date data, inside this "
                        "notebook (using the EXACT SAME VIF+p_max+OLS+LOOCV process as in "
                        "MLR_Analysis.ipynb) - instead of trying to read an "
                        "old/stale file. Therefore, there SHOULD BE no numerical "
                        "difference between the two reports (when run with the same "
                        "data).")
    add_paragraph(doc, "Per Reviewer 3's warning in item 8 (over-interpretation of "
                        "ANN/XGBoost results), this comparison is presented ONLY as a "
                        "numerical side-by-side - no strong claim is made that ANN is "
                        "'definitely superior' to MLR; on a small data set such as n=66, "
                        "small differences between the two methods may fall within the "
                        "margin of noise.", italic=True)

    doc.add_page_break()

## 9. Writing the Group Report (ANN)

Adds a group's FULL ANN result to the Word document: the hyperparameter performance ranking table, model performance (the mean±std of SEED_COUNT different seeds), Y-randomization, and the Williams Plot.

In [ ]:
def write_group_section(doc, seq_no, group_name, ann_result, mlr_result, sd):
    title = f"{seq_no}. {group_name}"
    add_heading(doc, title, level=1)

    print(f"\n{'='*70}\n{title}\n{'='*70}")
    architecture_text = 'x'.join(str(k) for k in ann_result['final_architecture'])
    print(f"ANN architecture={architecture_text}  alpha={ann_result['final_alpha']}")
    print(f"Q2LOO_ANN={ann_result['q2_loo_mean']:.3f}±{ann_result['q2_loo_std']:.3f}  "
          f"CCC_ANN={ann_result['ccc_mean']:.3f}±{ann_result['ccc_std']:.3f}")

    total_evaluations = ann_result['n'] * SEED_COUNT
    add_paragraph(doc, "Hyperparameter Performance Ranking (top 5 combinations by the "
                        f"MEAN of {ann_result['n']} outer rounds x {SEED_COUNT} repeats = "
                        f"{total_evaluations} evaluations of nested LOOCV)", bold=True)
    score_rows = []
    for (architecture, alpha), mean_score in ann_result['score_ranking'][:5]:
        architecture_text_row = 'x'.join(str(k) for k in architecture)
        score_rows.append([architecture_text_row, str(alpha), f"{mean_score:.4f}"])
    build_table(doc, ['Architecture (neurons)', 'Alpha', 'Mean Inner-CV R2'], score_rows)

    doc.add_paragraph()
    add_paragraph(doc, f"Selected (final) architecture: {architecture_text} neurons, alpha={ann_result['final_alpha']}", bold=True)
    add_paragraph(doc, f"The final model, when trained with {SEED_COUNT} different random seeds, "
                        f"converged on average in {ann_result['final']['mean_iteration']:.1f}"
                        f"±{ann_result['final']['std_iteration']:.1f} iterations "
                        f"(lbfgs solver).")

    add_paragraph(doc, "Model Performance — ANN  (S = RMSE, in the original unit; "
                        "%value = ratio of S to SD; all values are the mean±std of SEED_COUNT "
                        f"({SEED_COUNT}) different seeds)", bold=True)
    performance_rows = [[
        str(ann_result['n']),
        s_ratio_lines_mean(ann_result['final']['r2_fit_mean'], ann_result['final']['r2_fit_std'], ann_result['final']['s_fit_mean'], BP_UNIT),
        s_ratio_lines_mean(ann_result['q2_loo_mean'], ann_result['q2_loo_std'], ann_result['s_loo_mean'], BP_UNIT),
        s_ratio_lines_mean(ann_result['q2_k5_mean'], ann_result['q2_k5_std'], ann_result['s_k5_mean'], BP_UNIT),
        s_ratio_lines_mean(ann_result['q2_k10_mean'], ann_result['q2_k10_std'], ann_result['s_k10_mean'], BP_UNIT),
        f"{ann_result['ccc_mean']:.3f} ± {ann_result['ccc_std']:.3f}",
    ]]
    build_table(doc, ['n', 'R2_fit', 'Q2LOO', 'Q2(k=5)', 'Q2(k=10)', 'CCC'], performance_rows)
    doc.add_paragraph()

    add_paragraph(doc, "Y-Randomization Test — ANN  (both the actual model AND each random trial "
                        f"were computed at the same standard, for consistency, as the mean of {SEED_COUNT} "
                        "seeds)", bold=True)
    mean_random = float(np.mean(ann_result['random_r2s']))
    max_random = float(np.max(ann_result['random_r2s']))
    random_arr = np.asarray(ann_result['random_r2s'])
    actual_r2 = ann_result['actual_r2_yrand']
    exceed_fraction = float(np.mean(random_arr >= actual_r2))
    if exceed_fraction <= 0.05:
        yrand_text = (
            f"The ANN model's actual R2 value ({actual_r2:.3f}) was matched or exceeded by only "
            f"{exceed_fraction * 100:.1f}% of the {YRAND_REPEATS} random Y-shuffling trials "
            f"(random distribution mean={mean_random:.3f}, maximum={max_random:.3f}) — this "
            f"supports that the ANN model's correlation is NOT COINCIDENTAL."
        )
    elif actual_r2 > mean_random:
        yrand_text = (
            f"The ANN model's actual R2 value ({actual_r2:.3f}) is above the mean of the random "
            f"Y-shuffling trials (mean={mean_random:.3f}), but {exceed_fraction * 100:.1f}% of the "
            f"trials could still produce a value equal to or above it — this shows that the "
            f"separation between the model and chance is NOT STRONG, and this group's results "
            f"should be interpreted cautiously."
        )
    else:
        yrand_text = (
            f"The ANN model's actual R2 value ({actual_r2:.3f}) is AT OR BELOW the mean of the "
            f"random Y-shuffling trials (mean={mean_random:.3f}) — meaning the ANN model built "
            f"for this group is statistically INDISTINGUISHABLE from a coincidental correlation "
            f"and should NOT be accepted as a VALID/RELIABLE model."
        )
    add_paragraph(doc, yrand_text)
    english_title = GROUP_ENGLISH_NAME.get(group_name, group_name)
    yrand_fig = plot_yrand_figure(actual_r2, ann_result['random_r2s'], english_title)
    doc.add_picture(fig_to_bytes(yrand_fig), width=Inches(5.4))

    add_paragraph(doc, "Applicability Domain (Williams Plot) — ANN  (standardized residuals, "
                        f"computed from the mean outer predictions of {SEED_COUNT} seeds)", bold=True)
    outside_molecules = ann_result['outside_molecules']
    if outside_molecules:
        add_paragraph(doc, f"A total of {len(outside_molecules)} molecules fall outside the "
                            f"leverage (h*={ann_result['h_star']:.3f}) or standardized residual (±3σ) "
                            f"limits: {', '.join(map(str, outside_molecules))}.")
    else:
        add_paragraph(doc, "No molecules in this group fall outside the Applicability Domain.")
    williams_fig = plot_williams_plot(ann_result['leverage'], ann_result['standardized_residual'],
                                       ann_result['h_star'], ann_result['outside_molecules'], english_title)
    doc.add_picture(fig_to_bytes(williams_fig), width=Inches(5.4))

## 10. Main Run — bp ANN Analysis (No Options, Runs Directly)

Since this notebook is for a SINGLE property (bp), there is no menu like in the MLR notebook - when you run the cell below, ANN is trained for all 6 groups, compared with MLR, and a single Word report (`bp_ANN_Analysis.docx`) is written. **WARNING**: because nested cross-validation is repeated FROM SCRATCH with SEED_COUNT (10) different seeds, running this cell can take a VERY LONG TIME (on the order of hours) - this is a DELIBERATE choice, to eliminate the concern that a single-seed result could turn out lucky/unlucky (see Method Description).

In [ ]:
def run_bp_ann_analysis():
    print(f"\nStarting bp ANN analysis (can take a VERY LONG TIME, on the order of hours, "
          f"because nested cross-validation is repeated with {SEED_COUNT} seeds "
          "- this is a deliberate choice)...\n")
    properties_df, groups = load_data()

    doc = Document()
    add_heading(doc, "BP - BOILING POINT (°C) — ANN ANALYSIS", level=0)
    add_paragraph(doc, "Results Report (With MLR Comparison)", italic=True)

    ann_results = []
    mlr_results = []
    first_n, first_sd = None, None

    for i, group_name in enumerate(GROUP_SHEETS, start=1):
        print(f"[{i}/{len(GROUP_SHEETS)}] processing {group_name}...")
        molecules, x, y = prepare_bp_group_data(properties_df, groups, group_name)
        n = len(y)
        if n < 5:
            print(f"WARNING: not enough data for {group_name} (n={n}), skipping.")
            continue
        sd = calculate_std_dev(y)
        if first_n is None:
            first_n, first_sd = n, sd

        mlr_result = calculate_mlr_reference(x, y)
        mlr_result['group_name'] = group_name
        mlr_results.append(mlr_result)

        print(f"  -> calculating LOOCV ({SEED_COUNT} repeats)...")
        loo_multi = nested_cross_validation_multi_seed(x, y, lambda seed: LeaveOneOut())
        score_ranking = rank_by_mean_score(loo_multi['scores_pool'])
        final_architecture, final_alpha = score_ranking[0][0]

        q2_k5_mean = q2_k5_std = s_k5_mean = s_k5_std = None
        q2_k10_mean = q2_k10_std = s_k10_mean = s_k10_std = None
        if n >= K5_MIN_N:
            print(f"  -> calculating Q2(k=5) ({SEED_COUNT} repeats)...")
            k5_multi = nested_cross_validation_multi_seed(
                x, y, lambda seed: KFold(n_splits=5, shuffle=True, random_state=seed))
            q2_k5_mean, q2_k5_std = k5_multi['q2_mean'], k5_multi['q2_std']
            s_k5_mean, s_k5_std = k5_multi['s_mean'], k5_multi['s_std']
        if n >= K10_MIN_N:
            print(f"  -> calculating Q2(k=10) ({SEED_COUNT} repeats)...")
            k10_multi = nested_cross_validation_multi_seed(
                x, y, lambda seed: KFold(n_splits=10, shuffle=True, random_state=seed))
            q2_k10_mean, q2_k10_std = k10_multi['q2_mean'], k10_multi['q2_std']
            s_k10_mean, s_k10_std = k10_multi['s_mean'], k10_multi['s_std']

        print(f"  -> training final model ({SEED_COUNT} seeds)...")
        final_model_result = train_and_report_final_model(x, y, final_architecture, final_alpha)
        print(f"  -> running Y-Randomization ({YRAND_REPEATS} trials x {SEED_COUNT} seeds)...")
        actual_r2_yrand, random_r2s = apply_y_randomization_ann(
            x, y, final_architecture, final_alpha, actual_r2_list=final_model_result['r2_list'])

        s_outer_loo = calculate_q2_and_s(y, loo_multi['mean_prediction'])[1]
        leverage, standardized_residual, h_star, outside_mask = calculate_applicability_domain(
            x, CANDIDATE_VARIABLES, y, loo_multi['mean_prediction'], s_outer_loo
        )
        outside_molecules = [molecules.iloc[j] for j in np.where(outside_mask)[0]]

        ann_results.append({
            'group_name': group_name, 'n': n, 'score_ranking': score_ranking,
            'final_architecture': final_architecture, 'final_alpha': final_alpha,
            'q2_loo_mean': loo_multi['q2_mean'], 'q2_loo_std': loo_multi['q2_std'],
            's_loo_mean': loo_multi['s_mean'], 's_loo_std': loo_multi['s_std'],
            'ccc_mean': loo_multi['ccc_mean'], 'ccc_std': loo_multi['ccc_std'],
            'q2_k5_mean': q2_k5_mean, 'q2_k5_std': q2_k5_std,
            's_k5_mean': s_k5_mean, 's_k5_std': s_k5_std,
            'q2_k10_mean': q2_k10_mean, 'q2_k10_std': q2_k10_std,
            's_k10_mean': s_k10_mean, 's_k10_std': s_k10_std,
            'final': final_model_result,
            'actual_r2_yrand': actual_r2_yrand, 'random_r2s': random_r2s,
            'leverage': leverage, 'standardized_residual': standardized_residual,
            'h_star': h_star, 'outside_molecules': outside_molecules,
        })

    cover_rows = [
        ['Property Analyzed', 'bp - Boiling Point (°C)'],
        ['Total Number of Molecules (n)', str(first_n)],
        ["Data's Natural Standard Deviation (SD)", f"±{first_sd:.2f}{BP_UNIT}" if isinstance(first_sd, float) else str(first_sd)],
        ['Input Descriptors', ', '.join(CANDIDATE_VARIABLES) + ' (all, without elimination)'],
        ['Hyperparameter Search', f"{len(HIDDEN_LAYER_OPTIONS)} architectures x {len(ALPHA_OPTIONS)} alphas, "
                                   f"nested {INNER_KFOLD}-fold inner-CV, repeated with {SEED_COUNT} seeds"],
        ['Validation Steps', f'Nested LOOCV . k=5 (n>={K5_MIN_N}) . k=10 (n>={K10_MIN_N}) . '
                              f'Y-Randomization . CCC . Williams Plot (AD) — all as the mean±std of '
                              f'{SEED_COUNT} seeds'],
        ['Number of Groups Analyzed', str(len(ann_results))],
    ]
    build_table(doc, ['Information', 'Value'], cover_rows)
    doc.add_page_break()

    write_method_and_terms_section(doc)

    add_heading(doc, "Overall Comparison of Groups (ANN)", level=1)
    summary_rows = []
    for s in ann_results:
        architecture_text = 'x'.join(str(k) for k in s['final_architecture'])
        summary_rows.append([
            s['group_name'], architecture_text,
            s_ratio_lines_mean(s['final']['r2_fit_mean'], s['final']['r2_fit_std'], s['final']['s_fit_mean'], BP_UNIT),
            s_ratio_lines_mean(s['q2_loo_mean'], s['q2_loo_std'], s['s_loo_mean'], BP_UNIT),
            f"{s['ccc_mean']:.3f} ± {s['ccc_std']:.3f}",
        ])
    build_table(doc, ['Group', 'Architecture', 'R2_fit', 'Q2LOO', 'CCC'], summary_rows)
    doc.add_page_break()

    add_heading(doc, "Detailed Results by Group (ANN)", level=1)
    for i, s in enumerate(ann_results, start=1):
        write_group_section(doc, i, s['group_name'], s, mlr_results[i - 1], first_sd)

    doc.add_page_break()
    add_heading(doc, "MLR vs ANN Comparison", level=1)
    comparison_rows = []
    for ann_s, mlr_s in zip(ann_results, mlr_results):
        architecture_text = 'x'.join(str(k) for k in ann_s['final_architecture'])
        comparison_rows.append([
            ann_s['group_name'],
            ', '.join(mlr_s['variables']),
            s_ratio_lines(mlr_s['r2_fit'], mlr_s['s_fit'], BP_UNIT),
            s_ratio_lines(mlr_s['q2_loo'], mlr_s['s_loo'], BP_UNIT),
            f"{mlr_s['ccc']:.3f}",
            architecture_text,
            s_ratio_lines_mean(ann_s['final']['r2_fit_mean'], ann_s['final']['r2_fit_std'], ann_s['final']['s_fit_mean'], BP_UNIT),
            s_ratio_lines_mean(ann_s['q2_loo_mean'], ann_s['q2_loo_std'], ann_s['s_loo_mean'], BP_UNIT),
            f"{ann_s['ccc_mean']:.3f} ± {ann_s['ccc_std']:.3f}",
        ])
    build_table(doc, ['Group', 'MLR Variables', 'MLR R2_fit', 'MLR Q2LOO', 'MLR CCC',
                       'ANN Architecture', 'ANN R2_fit', 'ANN Q2LOO', 'ANN CCC'], comparison_rows)

    doc.add_page_break()
    add_heading(doc, "Overall Assessment", level=1)
    if ann_results and mlr_results:
        best_ann = max(ann_results, key=lambda s: s['q2_loo_mean'])
        best_mlr = max(mlr_results, key=lambda s: s['q2_loo'])
        add_paragraph(doc, f"For ANN, the highest Q2LOO value was observed in the {best_ann['group_name']} "
                            f"group (Q2LOO={best_ann['q2_loo_mean']:.3f}±{best_ann['q2_loo_std']:.3f}).")
        add_paragraph(doc, f"For MLR, the highest Q2LOO value was observed in the {best_mlr['group_name']} "
                            f"group (Q2LOO={best_mlr['q2_loo']:.3f}).")
        diff = best_ann['q2_loo_mean'] - best_mlr['q2_loo']
        if abs(diff) < 0.02:
            comparison_text = (
                "The Q2LOO values of the two methods' best groups are very close to each other "
                "(difference<0.02) - it cannot be said that ANN provides a meaningful advantage "
                "over MLR on this data set."
            )
        elif diff > 0:
            comparison_text = (
                f"ANN's best group gave a Q2LOO value {diff:.3f} higher than MLR's best group - "
                f"however, as Reviewer 3 warned (item 8), on a small data set such as n={first_n}, "
                f"it is NOT claimed that this difference is a generalizable/definitive advantage, "
                f"it is merely an observation on this data set."
            )
        else:
            comparison_text = (
                f"MLR's best group gave a Q2LOO value {abs(diff):.3f} higher than ANN's best group - "
                f"the simpler (linear) method performed at least as well (or better) on this data "
                f"set, without requiring ANN's added complexity."
            )
        add_paragraph(doc, comparison_text, italic=True)

    file_name = "bp_ANN_Analysis.docx"
    doc.save(file_name)
    print(f"\nReport saved: {file_name}")
    return ann_results, mlr_results


# Run the line below to execute (can take a VERY LONG TIME due to SEED_COUNT-fold repetition):
result = run_bp_ann_analysis()